# Sign Language Recognition: Publication-Ready Research Benchmark
### Complete Implementation of 4 Paradigms & Hardware Efficiency Profiling

This notebook reproduces the complete experimental suite from the paper:
1. **Baseline**: CNN Backbone + Bidirectional LSTM (`Baseline_CNN_BiLSTM`)
2. **Comparative**: Modern Lightweight Vision Backbone (`Comparative_Backbone`: ConvNeXt-Femto / MobileViT-S via `timm`)
3. **Proposed Model 1**: CNN + Pure-PyTorch Bidirectional Selective State Space Model (`Proposed1_CNN_Mamba`: Vision Mamba S6 with Zero-Order Hold Discretization)
4. **Proposed Model 2**: Lightweight Dual-Stream Adaptive Graph Convolutional Network + MobileNetV3 CNN with Gated Cross-Modal Fusion (`Proposed2_Lightweight_GCN_CNN`)

**Key Metrics Profiled:**
- Parameter Count ($M$)
- Theoretical FLOPs ($G$) via `thop`
- Real Inference Latency ($ms$) with 20 warmup + 100 timed passes
- Top-1 Test Accuracy (\%) & Macro-F1 (\%)
- Part 3: GCN Ablation Study (Fixed $\hat{\mathbf{A}}$ vs. Adaptive $\hat{\mathbf{A}} + \alpha \mathbf{A}_{ada}$ vs. Gated Fusion)
- Part 3: Adverse Illumination ($\gamma=0.3$) & Gaussian Noise ($\sigma=0.15$) Stress-Testing

## 1. Environment & Hardware Verification
Check GPU acceleration (e.g., NVIDIA T4, V100, or A100).

In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

## 2. Install Required Dependencies

In [ ]:
!pip install -q timm mediapipe opencv-python thop scikit-learn pandas matplotlib tabulate tqdm

## 3. Dataset Preparation
Unpack the 29-class ASL Alphabet dataset (`archive.zip`) if not already extracted.

In [ ]:
import os, zipfile

dataset_dir = './dataset_extracted/asl_alphabet_train/asl_alphabet_train'
if not os.path.exists(dataset_dir):
    if os.path.exists('archive.zip'):
        print('[*] Extracting archive.zip...')
        with zipfile.ZipFile('archive.zip', 'r') as zip_ref:
            zip_ref.extractall('./dataset_extracted')
        print('[*] Extraction complete.')
    else:
        print('[!] archive.zip not found in current directory. Please upload archive.zip or mount Google Drive.')
else:
    print(f'[*] Dataset already extracted at: {dataset_dir}')

classes = sorted(os.listdir(dataset_dir))
print(f'[*] Found {len(classes)} classes: {classes}')

## 4. Keypoint Preprocessing Verification
Verify that MediaPipe 21-joint landmark `.npy` cache files exist.

In [ ]:
import glob
npy_files = glob.glob(f'{dataset_dir}/**/*.npy', recursive=True)
print(f'[*] Total cached landmark files (.npy): {len(npy_files)}')
if len(npy_files) == 0:
    print('[*] Running offline landmark extraction with MediaPipe...')
    !python preprocess_keypoints.py --data_root {dataset_dir}
else:
    print('[*] Landmark cache verified. Ready for instant dual-modal training.')

## 5. Execute Full Unified Benchmark, Training & Profiling
Runs training for all 4 models, collects hardware efficiency statistics, conducts GCN ablations, and performs the noise/illumination stress-tests.

In [ ]:
# On a GPU, 5-10 epochs across all 4 models + ablations + stress-tests runs in ~5-10 minutes
!python train_benchmark.py --data_root {dataset_dir} --epochs 5 --batch_size 64

## 6. Publication Results & Tables

In [ ]:
import pandas as pd
from IPython.display import display

print('='*70)
print('TABLE I: MAIN HARDWARE EFFICIENCY & RECOGNITION BENCHMARK')
print('='*70)
df_bm = pd.read_csv('benchmark_results.csv')
display(df_bm)

print('\n' + '='*70)
print('TABLE II: PART 3 ABLATION STUDY ON PROPOSED MODEL 2 (GCN + CNN)')
print('='*70)
df_ab = pd.read_csv('ablation_results.csv')
display(df_ab)

print('\n' + '='*70)
print('TABLE III: PART 3 ILLUMINATION & NOISE ROBUSTNESS STRESS-TEST')
print('='*70)
df_st = pd.read_csv('stress_test_results.csv')
display(df_st)

## 7. Generate & Display Publication Figures

In [ ]:
!python generate_plots.py

from IPython.display import Image, display
print('Figure 1: Accuracy vs Parameter Count Trade-Off (Bubble Size ~ Latency)')
display(Image('fig_accuracy_vs_params.png'))

print('Figure 2: Robustness Under Adverse Illumination & Noise Stress')
display(Image('fig_robustness_stress.png'))

## 8. Export All Results
Bundle all generated CSV tables, figures, checkpoints, and LaTeX draft into a zip archive.

In [ ]:
!zip -r publication_bundle.zip *.csv *.png *.tex *.py
try:
    from google.colab import files
    files.download('publication_bundle.zip')
    print('[*] Download triggered successfully.')
except Exception as e:
    print('[*] Saved to publication_bundle.zip in working directory.')